## Initialisation des modules PyTorch et des bibliothèques pour l'entraînement distribué

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.utils.data import DataLoader, Dataset
import torch.distributed as dist
from torch.nn import functional as F
import os

## Implémentation du modèle GPT-2 simplifié

In [4]:
class GPT2(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer, n_head, seq_length):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=n_embd, nhead=n_head) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.embed(x) 
        for layer in self.layers:
            x = layer(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

## Création d'un jeu de données simulé

In [5]:
class OpenWebTextDataset(Dataset):
    def __init__(self, vocab_size, seq_length, num_samples):
        self.vocab_size = vocab_size
        self.seq_length = seq_length
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return torch.randint(0, self.vocab_size, (self.seq_length,))

## Configuration d'un Environnement Distribué et Boucle d'Entraînement avec FSDP et GPT-2

In [9]:
# Distributed environment setup
def setup_distributed():
    if not dist.is_initialized():
        os.environ.setdefault("RANK", "0")
        os.environ.setdefault("WORLD_SIZE", "1")
        os.environ.setdefault("LOCAL_RANK", "0")
        os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
        os.environ.setdefault("MASTER_PORT", "29500")
        dist.init_process_group("nccl")
        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))


def cleanup_distributed():
    dist.destroy_process_group()

# Training step
def train_step(model, batch, optimizer, vocab_size):
    batch = batch.cuda(non_blocking=True)
    outputs = model(batch)
    loss = F.cross_entropy(outputs.view(-1, vocab_size), batch.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

# Training loop
def train():
    cleanup_distributed()
    setup_distributed()

    # Hyperparameters
    vocab_size = 50257
    n_embd = 768
    n_layer = 12
    n_head = 12
    seq_length = 128
    batch_size = 8
    num_epochs = 2
    num_samples = 10000

    # Dataset and DataLoader
    dataset = OpenWebTextDataset(vocab_size, seq_length, num_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

    # Model and optimizer
    model = GPT2(vocab_size, n_embd, n_layer, n_head, seq_length).cuda()
    model = FSDP(model)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4)

    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch in dataloader:
            loss = train_step(model, batch, optimizer, vocab_size)
            epoch_loss += loss

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch + 1}, Average Loss: {avg_loss:.4f}")

    cleanup_distributed()

In [10]:
if __name__ == "__main__":
    train()

/usr/local/lib/python3.10/dist-packages/torch/distributed/fsdp/_init_utils.py:444: UserWarning: FSDP is switching to use `NO_SHARD` instead of ShardingStrategy.FULL_SHARD since the world size is 1.
  warnings.warn(


Epoch 1, Average Loss: 10.9205
Epoch 2, Average Loss: 10.8393


In [2]:
!pip install torch==2.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 8.7 MB/s eta 0:00:000:00:0100:01
   ━━